### Import libraries|

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

### Define project paths

In [2]:
PROJECT_ROOT = Path("..")

RAW_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
OUTPUT_PATH = PROJECT_ROOT / "outputs"

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

### Define months

In [3]:
MONTHS = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

MONTH_MAP = {
    "jan": "January",
    "january": "January",

    "feb": "February",
    "february": "February",

    "mar": "March",
    "march": "March",

    "apr": "April",
    "april": "April",

    "may": "May",

    "jun": "June",
    "june": "June",

    "jul": "July",
    "july": "July",

    "aug": "August",
    "august": "August",

    "sep": "September",
    "sept": "September",
    "september": "September",

    "oct": "October",
    "october": "October",

    "nov": "November",
    "november": "November",

    "dec": "December",
    "december": "December"
}

### Create Excel reader

In [4]:
def find_header_row(df):
    """
    Find the row containing the Country column.
    """
    
    for i in range(min(15, len(df))):
        row_values = df.iloc[i].astype(str).str.strip().str.lower()
        
        if row_values.str.contains("country", regex=False).any():
            return i
    
    raise ValueError("Country header row was not found.")

### Standardize month names

In [5]:
def standardize_month_name(column):
    value = str(column).strip().lower()
    
    return MONTH_MAP.get(value, None)

### Read one SLTDA file

In [6]:
def load_sltada_file(file_path):
    
    year = int(re.search(r"20\d{2}", file_path.name).group())
    
    raw = pd.read_excel(file_path, header=None)
    
    header_row = find_header_row(raw)
    
    df = pd.read_excel(
        file_path,
        header=header_row
    )

    df.columns = [
        str(col).strip()
        for col in df.columns
    ]

    country_col = None
    
    for col in df.columns:
        if str(col).strip().lower() == "country":
            country_col = col
            break
    
    if country_col is None:
        raise ValueError(f"Country column not found in {file_path.name}")
    
    month_columns = {}
    
    for col in df.columns:
        month = standardize_month_name(col)
        
        if month:
            month_columns[col] = month
    
    if len(month_columns) < 12:
        print(
            f"Warning: {file_path.name} "
            f"has {len(month_columns)} month columns."
        )
    
    selected_columns = [country_col] + list(month_columns.keys())
    
    df = df[selected_columns].copy()
    
    rename_dict = {
        country_col: "Country"
    }
    
    rename_dict.update(month_columns)
    
    df = df.rename(columns=rename_dict)
 
    df["Country"] = df["Country"].astype(str).str.strip()
    
    df = df[
        ~df["Country"].str.lower().isin(
            ["total", "nan", ""]
        )
    ].copy()
    
    for month in MONTHS:
        if month in df.columns:
            df[month] = pd.to_numeric(
                df[month],
                errors="coerce"
            )
    
    df["Year"] = year
    
    df = df[
        df["Country"].notna()
    ].copy()
    
    return df

### Load all files

In [7]:
all_files = sorted(RAW_PATH.glob("SLTDA_*.xlsx"))

print("Files found:")

for file in all_files:
    print(file.name)

Files found:
SLTDA_2018.xlsx
SLTDA_2019.xlsx
SLTDA_2020.xlsx
SLTDA_2021.xlsx
SLTDA_2022.xlsx
SLTDA_2023.xlsx
SLTDA_2024.xlsx
SLTDA_2025.xlsx


### Read all years

In [8]:
yearly_data = []

for file in all_files:
    
    try:
        df_year = load_sltada_file(file)
        
        yearly_data.append(df_year)
        
        print(
            f"{file.name}: "
            f"{len(df_year)} countries loaded"
        )
        
    except Exception as e:
        print(
            f"ERROR in {file.name}: {e}"
        )

SLTDA_2018.xlsx: 197 countries loaded
SLTDA_2019.xlsx: 198 countries loaded
SLTDA_2020.xlsx: 191 countries loaded
SLTDA_2021.xlsx: 214 countries loaded
SLTDA_2022.xlsx: 192 countries loaded
SLTDA_2023.xlsx: 190 countries loaded
SLTDA_2024.xlsx: 193 countries loaded
SLTDA_2025.xlsx: 200 countries loaded


### Combine datasets

In [9]:
tourism_wide = pd.concat(
    yearly_data,
    ignore_index=True
)

print("Shape:", tourism_wide.shape)

tourism_wide.head()

Shape: (1575, 14)


,Country,January,February,March,April,May,June,July,August,September,October,November,December,Year
0,"AFGHANISTAN, ISLAMIC REPUBLIC OF",83.0,150.0,112.0,40.0,108.0,44.0,47.0,78.0,34.0,41.0,52.0,72.0,2018
1,"ALBANIA, REPUBLIC OF",14.0,15.0,5.0,16.0,12.0,15.0,11.0,19.0,14.0,13.0,12.0,31.0,2018
2,"ALGERIA, PEOPLE'S DEMOCRATIC REPUBLIC OF",38.0,35.0,28.0,24.0,12.0,19.0,24.0,71.0,27.0,20.0,27.0,43.0,2018
3,"ANDORRA, PRINCIPALITY OF",4.0,6.0,2.0,3.0,6.0,0.0,5.0,16.0,3.0,3.0,1.0,4.0,2018
4,"ANGOLA, REPUBLIC OF",0.0,1.0,2.0,1.0,0.0,0.0,0.0,0.0,2.0,1.0,1.0,1.0,2018


### Convert Wide → Long

In [10]:
tourism_long = tourism_wide.melt(
    id_vars=["Year", "Country"],
    value_vars=MONTHS,
    var_name="Month",
    value_name="Tourist_Arrivals"
)

tourism_long.head()

,Year,Country,Month,Tourist_Arrivals
0,2018,"AFGHANISTAN, ISLAMIC REPUBLIC OF",January,83.0
1,2018,"ALBANIA, REPUBLIC OF",January,14.0
2,2018,"ALGERIA, PEOPLE'S DEMOCRATIC REPUBLIC OF",January,38.0
3,2018,"ANDORRA, PRINCIPALITY OF",January,4.0
4,2018,"ANGOLA, REPUBLIC OF",January,0.0


### Clean country names

In [11]:
tourism_long["Country"] = (
    tourism_long["Country"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

tourism_long["Tourist_Arrivals"] = pd.to_numeric(
    tourism_long["Tourist_Arrivals"],
    errors="coerce"
)

tourism_long["Tourist_Arrivals"] = (
    tourism_long["Tourist_Arrivals"]
    .fillna(0)
)

### Add date

In [12]:
month_number = {
    month: i
    for i, month in enumerate(MONTHS, start=1)
}

tourism_long["Month_Number"] = (
    tourism_long["Month"]
    .map(month_number)
)

tourism_long["Date"] = pd.to_datetime(
    tourism_long["Year"].astype(str)
    + "-"
    + tourism_long["Month_Number"].astype(str)
    + "-01"
)

### Sort data

In [13]:
tourism_long = tourism_long.sort_values(
    ["Date", "Country"]
).reset_index(drop=True)

### Save master dataset

In [14]:
MASTER_FILE = (
    PROCESSED_PATH /
    "01_tourism_arrivals_clean.csv"
)

tourism_long.to_csv(
    MASTER_FILE,
    index=False
)

print("Saved:", MASTER_FILE)

Saved: ..\data\processed\01_tourism_arrivals_clean.csv


### Create summary datasets

In [15]:
yearly_summary = (
    tourism_long
    .groupby("Year", as_index=False)
    ["Tourist_Arrivals"]
    .sum()
)

monthly_summary = (
    tourism_long
    .groupby(
        ["Year", "Month_Number", "Month"],
        as_index=False
    )["Tourist_Arrivals"]
    .sum()
)

country_summary = (
    tourism_long
    .groupby("Country", as_index=False)
    ["Tourist_Arrivals"]
    .sum()
)

country_month_summary = (
    tourism_long
    .groupby(
        ["Country", "Month"],
        as_index=False
    )["Tourist_Arrivals"]
    .sum()
)

### Save summaries

In [16]:
yearly_summary.to_csv(
    PROCESSED_PATH / "yearly_summary.csv",
    index=False
)

monthly_summary.to_csv(
    PROCESSED_PATH / "monthly_summary.csv",
    index=False
)

country_summary.to_csv(
    PROCESSED_PATH / "country_summary.csv",
    index=False
)

country_month_summary.to_csv(
    PROCESSED_PATH / "country_month_summary.csv",
    index=False
)

print("All processed files saved.")

All processed files saved.
